# Lesson 2.4 — AWS Systems Manager Automation & Runbooks

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Navigate the key SSM capabilities: Session Manager, Parameter Store, Patch Manager, Run Command
- Execute SSM Run Command to run scripts on EC2 instances without SSH
- Store and retrieve secrets/config using SSM Parameter Store
- Create and run an SSM Automation document (runbook)
- Use SSM Inventory to query fleet-wide instance state

**Exam relevance:** SOA-C03 Domain 1 (Skill 1.2.3), Domain 3 (Skill 3.2.1)

_Reference: [AWS Systems Manager Docs](https://docs.aws.amazon.com/systems-manager/latest/userguide/)_

## 1. AWS Systems Manager (SSM) Overview

**SSM** is AWS's fleet management service — it lets you manage EC2 instances (and on-premises servers) **without SSH or RDP**.

### Key SSM Capabilities

| Capability | What it does | Exam relevance |
|-----------|-------------|---------------|
| **Session Manager** | Interactive shell access without SSH/port 22 | Security — no inbound ports needed |
| **Run Command** | Execute scripts/commands fleet-wide | Operational tasks at scale |
| **Parameter Store** | Secure configuration and secrets storage | Config management |
| **Patch Manager** | Automated OS patching with approval workflows | Compliance |
| **Automation** | Multi-step workflow runbooks | Complex operational tasks |
| **Inventory** | Collect software/config data from instances | Fleet visibility |
| **OpsCenter** | Aggregate operational issues (OpsItems) | Incident management |
| **Maintenance Windows** | Schedule tasks during defined windows | Change management |

### SSM Agent
The **SSM Agent** is pre-installed on most AWS managed AMIs (Amazon Linux 2, Windows Server, Ubuntu). The instance needs:
1. SSM Agent running
2. IAM role with `AmazonSSMManagedInstanceCore` policy
3. Network access to SSM endpoints (via internet or VPC endpoint)

### Parameter Store: Standard vs Advanced
| Feature | Standard | Advanced |
|---------|---------|----------|
| Max size | 4KB | 8KB |
| Cost | Free | $0.05/param/month |
| Parameter policies | No | Yes (expiry, notifications) |
| Max parameters | 10,000 | Unlimited |

In [ ]:
import boto3
import json
import time

ssm = boto3.client('ssm', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']
REGION = 'us-east-1'

print(f"Account: {account_id} | Region: {REGION}")

In [ ]:
# SSM Parameter Store — store application config and secrets
# Best practice: use a path hierarchy for organization

# Store parameters in a path hierarchy
parameters = [
    {
        'name': '/cloudops-learning/dev/database/host',
        'value': 'myapp-dev.cluster-xyz.us-east-1.rds.amazonaws.com',
        'type': 'String',
        'description': 'RDS cluster endpoint for dev environment'
    },
    {
        'name': '/cloudops-learning/dev/database/port',
        'value': '5432',
        'type': 'String',
        'description': 'PostgreSQL port'
    },
    {
        'name': '/cloudops-learning/dev/app/debug-mode',
        'value': 'true',
        'type': 'String',
        'description': 'Enable debug logging'
    },
    {
        'name': '/cloudops-learning/dev/secrets/db-password',
        'value': 'SuperSecretPassword123!',  # In practice, use Secrets Manager for passwords
        'type': 'SecureString',               # Encrypted with KMS
        'description': 'Database password (use Secrets Manager in production!)'
    }
]

for param in parameters:
    ssm.put_parameter(
        Name=param['name'],
        Value=param['value'],
        Type=param['type'],
        Description=param['description'],
        Overwrite=True
    )
    print(f"Stored: {param['name']} ({param['type']})")

In [ ]:
# Retrieve parameters by path — get all config for an environment at once
response = ssm.get_parameters_by_path(
    Path='/cloudops-learning/dev/',
    Recursive=True,
    WithDecryption=True   # Decrypt SecureString params
)

print("Parameters in /cloudops-learning/dev/:")
config = {}
for param in response['Parameters']:
    name = param['Name'].split('/')[-1]  # Get last path component as key
    config[name] = param['Value']
    # Mask SecureString values in output
    display_value = '***MASKED***' if 'secret' in param['Name'] else param['Value']
    print(f"  {param['Name']:55} = {display_value}")

print(f"\nLoaded {len(config)} configuration values")

In [ ]:
# SSM Run Command — execute commands on instances without SSH
# This requires managed instances (SSM Agent running + IAM role)
# If you have no managed instances, this cell shows the structure but won't execute

# List managed instances
managed_instances = ssm.describe_instance_information()['InstanceInformationList']

if managed_instances:
    instance_id = managed_instances[0]['InstanceId']
    print(f"Running command on: {instance_id}")
    
    # Run a shell command on the instance
    response = ssm.send_command(
        InstanceIds=[instance_id],
        DocumentName='AWS-RunShellScript',
        Parameters={
            'commands': [
                'echo "=== System Info ==="',
                'uptime',
                'df -h /',
                'free -m',
                'cat /etc/os-release | grep PRETTY'
            ]
        },
        TimeoutSeconds=60,
        Comment='CloudOps health check'
    )
    
    command_id = response['Command']['CommandId']
    print(f"Command ID: {command_id}")
    
    # Wait for command to complete
    time.sleep(5)
    output = ssm.get_command_invocation(
        CommandId=command_id,
        InstanceId=instance_id
    )
    print(f"Status: {output['Status']}")
    print(f"Output:\n{output['StandardOutputContent']}")
else:
    print("No managed SSM instances found.")
    print("To use Run Command: Launch EC2 with AmazonSSMManagedInstanceCore role")
    print("\nExample Run Command structure:")
    print("""
ssm.send_command(
    InstanceIds=['i-1234567890abcdef0'],
    DocumentName='AWS-RunShellScript',
    Parameters={'commands': ['echo hello']}
)""")

In [ ]:
# Create a custom SSM Automation Document (Runbook)
# Runbooks automate multi-step operational tasks
# This example: automated EC2 instance restart with pre/post health checks

automation_document = {
    "schemaVersion": "0.3",
    "description": "Automated EC2 instance restart with health verification",
    "parameters": {
        "InstanceId": {
            "type": "String",
            "description": "EC2 Instance ID to restart"
        },
        "AutomationAssumeRole": {
            "type": "String",
            "description": "IAM role ARN for automation to assume",
            "default": ""
        }
    },
    "mainSteps": [
        {
            "name": "verifyInstanceRunning",
            "action": "aws:assertAwsResourceProperty",
            "inputs": {
                "Service": "ec2",
                "Api": "DescribeInstances",
                "InstanceIds": ["{{ InstanceId }}"],
                "PropertySelector": "$.Reservations[0].Instances[0].State.Name",
                "DesiredValues": ["running"]
            }
        },
        {
            "name": "stopInstance",
            "action": "aws:changeInstanceState",
            "inputs": {
                "InstanceIds": ["{{ InstanceId }}"],
                "DesiredState": "stopped"
            }
        },
        {
            "name": "startInstance",
            "action": "aws:changeInstanceState",
            "inputs": {
                "InstanceIds": ["{{ InstanceId }}"],
                "DesiredState": "running"
            }
        },
        {
            "name": "waitForInstanceOnline",
            "action": "aws:waitForAwsResourceProperty",
            "timeoutSeconds": 300,
            "inputs": {
                "Service": "ssm",
                "Api": "DescribeInstanceInformation",
                "Filters": [{"Key": "InstanceIds", "Values": ["{{ InstanceId }}"]}],
                "PropertySelector": "$.InstanceInformationList[0].PingStatus",
                "DesiredValues": ["Online"]
            }
        }
    ]
}

try:
    response = ssm.create_document(
        Name='CloudOps-RestartInstanceWithHealthCheck',
        Content=json.dumps(automation_document),
        DocumentType='Automation',
        DocumentFormat='JSON'
    )
    print("Created SSM Automation Document: CloudOps-RestartInstanceWithHealthCheck")
    print(f"Status: {response['DocumentDescription']['Status']}")
except ssm.exceptions.DocumentAlreadyExists:
    print("Document already exists: CloudOps-RestartInstanceWithHealthCheck")

print("\nThis runbook can be triggered:")
print("  - Manually from the console")
print("  - Via EventBridge when a CloudWatch Alarm fires")
print("  - Via the API: ssm.start_automation_execution(DocumentName='...', Parameters={...})")

In [ ]:
# List useful predefined AWS Automation Documents
paginator = ssm.get_paginator('list_documents')

aws_docs = []
for page in paginator.paginate(
    Filters=[{'Key': 'Owner', 'Values': ['Amazon']}]
):
    aws_docs.extend(page['DocumentIdentifiers'])

# Show Automation documents (not Command documents)
automation_docs = [d for d in aws_docs if d.get('DocumentType') == 'Automation']
print(f"AWS-owned Automation documents ({len(automation_docs)} total). Useful examples:")

useful = [
    'AWS-RestartEC2Instance',
    'AWS-StartEC2Instance', 
    'AWS-StopEC2Instance',
    'AWS-CreateImage',
    'AWS-PatchInstanceWithRollback',
    'AWS-EnableCloudTrail',
    'AWS-DisablePublicAccessForSecurityGroup',
    'AWSSupport-TroubleshootRDPConnection'
]

available = {d['Name'] for d in automation_docs}
for doc_name in useful:
    status = 'Available' if doc_name in available else 'Not found'
    print(f"  [{status:12}] {doc_name}")

## Exercise

In [ ]:
# TODO: Store and retrieve application configuration using Parameter Store
# 1. Store these parameters under the path /cloudops-learning/prod/:
#    - /cloudops-learning/prod/app/max-connections (value: '100', type: String)
#    - /cloudops-learning/prod/app/log-level (value: 'WARN', type: String)
#    - /cloudops-learning/prod/cache/redis-host (value: 'redis.prod.internal', type: String)
#
# 2. Retrieve all parameters under /cloudops-learning/prod/ by path
# 3. Print them in a 'KEY=VALUE' format suitable for an .env file

import boto3
ssm = boto3.client('ssm', region_name='us-east-1')

# TODO: Store parameters


# TODO: Retrieve and format as .env file
def get_env_config(path_prefix: str) -> str:
    """Retrieve SSM parameters and return as .env file content."""
    # YOUR CODE HERE
    pass

env_content = get_env_config('/cloudops-learning/prod/')
print(env_content)

In [ ]:
# Cleanup
import boto3

ssm = boto3.client('ssm', region_name='us-east-1')

# Delete all parameters in our learning paths
for path in ['/cloudops-learning/dev/', '/cloudops-learning/prod/']:
    try:
        response = ssm.get_parameters_by_path(Path=path, Recursive=True)
        param_names = [p['Name'] for p in response['Parameters']]
        if param_names:
            ssm.delete_parameters(Names=param_names)
            print(f"Deleted {len(param_names)} parameters from {path}")
    except Exception as e:
        print(f"{path}: {e}")

# Delete the custom document
try:
    ssm.delete_document(Name='CloudOps-RestartInstanceWithHealthCheck')
    print("Deleted SSM document")
except Exception as e:
    print(f"Document: {e}")

## Summary

- **SSM Session Manager** replaces SSH — no port 22 needed, sessions are logged
- **Run Command** executes scripts fleet-wide using `AWS-RunShellScript` or `AWS-RunPowerShellScript`
- **Parameter Store** stores config (String) and secrets (SecureString, encrypted by KMS). Use path hierarchies: `/app/env/key`
- **Automation Documents** (runbooks) orchestrate multi-step operational workflows
- Pre-built AWS documents handle common tasks: restart, patch, create AMI

## Key Exam Points (SOA-C03)
- Parameter Store SecureString = **KMS encryption** (uses `aws/ssm` key by default)
- SSM Agent requires **outbound HTTPS** to SSM endpoints (or VPC endpoint for private instances)
- Automation documents can be triggered by **EventBridge** (CloudWatch Alarm → EventBridge → SSM Automation)
- **Maintenance Windows** schedule SSM tasks during defined periods (patch window)

---

**Next:** [Lesson 2.5 — Performance Optimization: EC2, EBS, RDS, S3](lesson-2.5-performance-optimization.ipynb)